# Hyperparameter Tuning — Optuna sobre LightGBM
60 trials de búsqueda bayesiana (TPE) con 3-fold CV.
Compara el modelo tuneado contra el baseline de `train_06.py`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.models.tune_07 import main, plot_tuning_history, N_TRIALS
from src.models.train_06 import load_data, evaluate_on_test, plot_roc_pr, MODELS_DIR

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
from sklearn.model_selection import train_test_split
sns.set_theme(style='whitegrid')
%matplotlib inline
RANDOM_STATE = 42

## 1. Ejecutar tuning

In [ ]:
tuned_model, study = main()

## 2. Historial de optimización

In [ ]:
plot_tuning_history(study)

## 3. Mejores parámetros

In [ ]:
trials = study.trials_dataframe().sort_values('value', ascending=False)
print(f'Mejor AUC CV: {study.best_value:.5f}')
print('\nMejores parámetros:')
for k, v in study.best_params.items():
    print(f'  {k:25s}: {v}')
trials.head(10)

## 4. Importancia de parámetros

In [ ]:
importances = optuna.importance.get_param_importances(study)
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(list(importances.keys())[::-1], list(importances.values())[::-1],
        color='#2196F3', edgecolor='white')
ax.set_title('Importancia de hiperparámetros (Optuna FAnova)')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

## 5. Cargar modelos y comparar en test

In [ ]:
X, y = load_data()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

baseline = joblib.load(MODELS_DIR / 'best_model.joblib')['model']
tuned    = joblib.load(MODELS_DIR / 'lgbm_tuned.joblib')['model']

results = [
    evaluate_on_test('LightGBM baseline', baseline, X_test, y_test),
    evaluate_on_test('LightGBM tuned',    tuned,    X_test, y_test),
]
plot_roc_pr(results, y_test)